# Sleep DataLoader Usage Example

This notebook demonstrates how to use the `sleep_dataloader` module to create train/val/test dataloaders for sleep EEG data with subject-based splitting.

## 1. Setup and Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from braindecode.datasets import SleepPhysionet
from braindecode.preprocessing import preprocess, Preprocessor
from braindecode.preprocessing.windowers import create_windows_from_events
from eeg_music.sleep_dataloader import create_sleep_dataloaders

## 2. Configuration

In [3]:
# Dataset configuration
SUBJECT_IDS = list(range(5))  # Use first 5 subjects for quick demo
RECORDING_IDS = [1]  # First night recordings

# Window configuration
WINDOW_SIZE_S = 30  # 30-second windows
SFREQ = 100  # Resample to 100 Hz
WINDOW_SIZE_SAMPLES = WINDOW_SIZE_S * SFREQ

# DataLoader configuration
N_CHANNELS = 2  # Number of EEG channels
BATCH_SIZE = 32
RANDOM_STATE = 42

## 3. Load Sleep Physionet Dataset

In [4]:
dataset = SleepPhysionet(
    subject_ids=SUBJECT_IDS,
    recording_ids=RECORDING_IDS,
    crop_wake_mins=30  # Crop 30 minutes of wake before/after sleep
)

print(f"Loaded {len(dataset)} recordings")
print(f"Subjects: {[ds.description['subject'] for ds in dataset.datasets]}")

Using default location ~/mne_data for PHYSIONET_SLEEP...


100%|█████████████████████████████████████| 48.3M/48.3M [00:00<00:00, 73.9GB/s]
100%|█████████████████████████████████████| 4.62k/4.62k [00:00<00:00, 15.1MB/s]
100%|█████████████████████████████████████| 51.1M/51.1M [00:00<00:00, 91.8GB/s]
100%|█████████████████████████████████████| 3.90k/3.90k [00:00<00:00, 8.23MB/s]
100%|█████████████████████████████████████| 51.1M/51.1M [00:00<00:00, 78.3GB/s]
100%|█████████████████████████████████████| 4.80k/4.80k [00:00<00:00, 14.3MB/s]
100%|█████████████████████████████████████| 51.4M/51.4M [00:00<00:00, 94.9GB/s]
100%|█████████████████████████████████████| 3.70k/3.70k [00:00<00:00, 10.7MB/s]
100%|█████████████████████████████████████| 46.9M/46.9M [00:00<00:00, 67.1GB/s]
100%|█████████████████████████████████████| 4.83k/4.83k [00:00<00:00, 9.01MB/s]

Download complete in 14m09s (237.4 MB)
Extracting EDF parameters from /home/zmrocze/mne_data/physionet-sleep-data/SC4001E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...


Extracting EDF parameters from /home/zmrocze/mne_data/physionet-sleep-data/SC4011E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from /home/zmrocze/mne_data/physionet-sleep-data/SC4021E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from /home/zmrocze/mne_data/physionet-sleep-data/SC4031E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from /home/zmrocze/mne_data/physionet-sleep-data/SC4041E0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Loaded 15384005 recordings
Subjects: [0, 1, 2, 3, 4]


## 4. Preprocess the Data

In [5]:
preprocessors = [
    Preprocessor('filter', l_freq=0.5, h_freq=30),  # Bandpass filter
    Preprocessor('resample', sfreq=SFREQ)  # Resample to 100 Hz
]

preprocess(dataset, preprocessors)
print("Preprocessing complete")

Reading 0 ... 2508000  =      0.000 ... 25080.000 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 661 samples (6.610 s)

Sampling frequency of the instance is already 100.0, returning unmodified.


/home/zmrocze/studia/uwr/eeg-magisterka/.devenv/state/venv/lib/python3.12/site-packages/braindecode/preprocessing/preprocess.py:76: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(


Reading 0 ... 3261000  =      0.000 ... 32610.000 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 661 samples (6.610 s)

Sampling frequency of the instance is already 100.0, returning unmodified.
Reading 0 ... 3060000  =      0.000 ... 30600.000 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamm

## 5. Create Windows from Events

We create 30-second non-overlapping windows and map sleep stages to numeric labels:
- 0: Wake
- 1: N1 (Stage 1)
- 2: N2 (Stage 2)
- 3: N3 (Stage 3/4 merged)
- 4: REM

In [6]:
# Sleep stage mapping (merging stages 3 and 4 following AASM standards)
mapping = {
    "Sleep stage W": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,
    "Sleep stage R": 4,
}

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=WINDOW_SIZE_SAMPLES,
    window_stride_samples=WINDOW_SIZE_SAMPLES,
    preload=True,
    mapping=mapping,
)

print(f"Created {len(windows_dataset)} windows")
print(f"Window shape: {windows_dataset[0][0].shape}")

Created 5132 windows
Window shape: (2, 3000)


## 6. Create DataLoaders with Subject-Based Splitting

The key feature: subjects are split into train/val/test, not individual windows. This prevents data leakage.

In [7]:
train_loader, val_loader, test_loader = create_sleep_dataloaders(
    windows_dataset=windows_dataset,
    window_length=WINDOW_SIZE_SAMPLES,
    n_channels=N_CHANNELS,
    batch_size=BATCH_SIZE,
    test_size=0.4,  # 40% of subjects for test+val
    val_split=0.5,  # Split test+val equally -> 20% val, 20% test
    random_state=RANDOM_STATE,
    num_workers=0
)

Train subjects: 3, samples: 2809
Val subjects: 1, samples: 1088
Test subjects: 1, samples: 1235


In [8]:
from eeg_music.sleep_dataloader import load_and_create_sleep_dataloaders
train_loader, val_loader, test_loader = load_and_create_sleep_dataloaders(
    subject_ids=list(range(5)),  # Load first 5 subjects
    crop_wake_mins=30,  # Crop 30 min of wake before/after sleep
    window_size_s=30,  # 30-second windows
    sfreq=100,  # Resample to 100 Hz
    n_channels=2,  # Use 2 EEG channels
    batch_size=32,
    test_size=0.4,  # 40% subjects for test+val
    val_split=0.5,  # Split test+val equally
    random_state=42,
    num_workers=0,
    l_freq=0.5,  # Bandpass filter 0.5-30 Hz
    h_freq=30.0,
)

print("\n=== Dataloader Summary ===")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Examine a batch
windows, labels, subject_ids = next(iter(train_loader))
print(f"\n=== Batch Structure ===")
print(f"Windows shape: {windows.shape}")  # (batch, channels, time)
print(f"Labels shape: {labels.shape}")  # (batch,)
print(f"Unique labels: {sorted(set(labels.tolist()))}")  # Sleep stages 0-4
print(f"Subject IDs in batch: {set(subject_ids)}")


ImportError: cannot import name 'load_and_create_sleep_dataloaders' from 'eeg_music.sleep_dataloader' (/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/sleep_dataloader.py)

## 7. Inspect DataLoader Properties

In [ ]:
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"\nTotal train samples: {len(train_loader.dataset)}")
print(f"Total val samples: {len(val_loader.dataset)}")
print(f"Total test samples: {len(test_loader.dataset)}")

## 8. Examine a Sample Batch

In [ ]:
# Get one batch from training set
windows, labels, subject_ids = next(iter(train_loader))

print(f"Windows shape: {windows.shape}")
print(f"  - Batch size: {windows.shape[0]}")
print(f"  - Channels: {windows.shape[1]}")
print(f"  - Time samples: {windows.shape[2]}")
print(f"\nLabels shape: {labels.shape}")
print(f"Labels: {labels[:10].tolist()}")
print(f"\nUnique subjects in batch: {len(set(subject_ids))}")
print(f"Subject IDs: {list(set(subject_ids))}")

## 9. Visualize Label Distribution

In [ ]:
stage_names = ['Wake', 'N1', 'N2', 'N3', 'REM']
stage_counts = [(labels == i).sum().item() for i in range(5)]

plt.figure(figsize=(10, 5))
plt.bar(stage_names, stage_counts, color=['orange', 'skyblue', 'green', 'purple', 'red'])
plt.xlabel('Sleep Stage')
plt.ylabel('Count')
plt.title('Sleep Stage Distribution in Sample Batch')
plt.grid(axis='y', alpha=0.3)
for i, count in enumerate(stage_counts):
    plt.text(i, count, str(count), ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 10. Visualize Sample EEG Windows

In [ ]:
# Plot first 3 windows from the batch
fig, axes = plt.subplots(3, 2, figsize=(15, 8))
time = np.arange(WINDOW_SIZE_SAMPLES) / SFREQ

for i in range(3):
    for ch in range(2):
        axes[i, ch].plot(time, windows[i, ch, :].numpy())
        axes[i, ch].set_xlabel('Time (s)')
        axes[i, ch].set_ylabel('Amplitude (µV)')
        axes[i, ch].set_title(f'Sample {i+1}, Channel {ch+1} - {stage_names[labels[i]]}')
        axes[i, ch].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Verify Subject-Based Splitting

Check that subjects don't overlap between train/val/test sets

In [ ]:
def get_all_subjects(loader):
    subjects = set()
    for _, _, subject_ids in loader:
        subjects.update(subject_ids)
    return subjects

train_subjects = get_all_subjects(train_loader)
val_subjects = get_all_subjects(val_loader)
test_subjects = get_all_subjects(test_loader)

print(f"Train subjects: {sorted(train_subjects)}")
print(f"Val subjects: {sorted(val_subjects)}")
print(f"Test subjects: {sorted(test_subjects)}")

# Check for overlap
train_val_overlap = train_subjects & val_subjects
train_test_overlap = train_subjects & test_subjects
val_test_overlap = val_subjects & test_subjects

print(f"\nTrain-Val overlap: {train_val_overlap if train_val_overlap else 'None ✓'}")
print(f"Train-Test overlap: {train_test_overlap if train_test_overlap else 'None ✓'}")
print(f"Val-Test overlap: {val_test_overlap if val_test_overlap else 'None ✓'}")

## 12. Example Training Loop Structure

In [ ]:
import torch
import torch.nn as nn

# Pseudo-code for training (not executed)
print("""
Example training loop:

# Create model
from eegnet import EEGNet
model = EEGNet(n_classes=5, n_channels=2, input_window_samples=3000)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    for windows, labels, subject_ids in train_loader:
        optimizer.zero_grad()
        outputs = model(windows)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for windows, labels, subject_ids in val_loader:
            outputs = model(windows)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f'Epoch {epoch+1}: Train Loss={train_loss/len(train_loader):.4f}, '
          f'Val Loss={val_loss/len(val_loader):.4f}, '
          f'Val Acc={100*correct/total:.2f}%')
""")

## Summary

This notebook demonstrated:

1. ✓ Loading Sleep Physionet dataset
2. ✓ Preprocessing (filtering, resampling)
3. ✓ Creating windows with sleep stage labels
4. ✓ Creating train/val/test dataloaders with **subject-based splitting**
5. ✓ Inspecting batch contents (windows, labels, subject IDs)
6. ✓ Visualizing data and label distributions
7. ✓ Verifying no subject overlap between splits

The dataloaders are now ready to use for training sleep stage classification models!